In [ ]:
#
# Purpose: provide a single function `openStream` that opens the default webcam and
# returns an OpenCV VideoCapture object. Keep this function pure in responsibility
# (only opens and configures the capture device).

import cv2
from typing import Iterator, Tuple, Optional
import numpy as np
#


def openStream(cam_index: int = 0, width: Optional[int] = None, height: Optional[int] = None, backend: Optional[int] = None) -> cv2.VideoCapture:
    """Open and configure a webcam stream using OpenCV.

    This function performs exactly one task: open a cv2.VideoCapture for the
    provided camera index and optionally set the capture resolution.

    Parameters
    ----------
    cam_index : int
        Index of the camera device (default 0).
    width : Optional[int]
        Desired capture width in pixels. If provided, the function will attempt
        to set the camera property. This is a best-effort; not all cameras
        support arbitrary resolutions.
    height : Optional[int]
        Desired capture height in pixels.
    backend : Optional[int]
        Optional OpenCV backend flag (e.g., cv2.CAP_DSHOW on Windows) to force.

    Returns
    -------
    cv2.VideoCapture
        An opened VideoCapture object. The caller is responsible for checking
        `cap.isOpened()` and for releasing the capture when finished
        (`cap.release()`).

    Raises
    ------
    RuntimeError
        If the capture device cannot be opened.

    Notes
    -----
    Keep this function single-responsibility: it opens and configures the
    capture device and returns it. Reading frames, preprocessing, visualization,
    and logging are separate functions to be implemented later.
    """
    # Choose backend if provided
    if backend is not None:
        cap = cv2.VideoCapture(cam_index, backend)
    else:
        cap = cv2.VideoCapture(cam_index)

    # Check if opened
    if not cap.isOpened():
        # Try a more forceful open (useful on some platforms)
        cap.open(cam_index)

    if not cap.isOpened():
        raise RuntimeError(f"Unable to open webcam (index={cam_index}). Check that the camera is connected and not used by another process.")

    # Try to set the resolution if requested (best-effort)
    if width is not None:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))
    if height is not None:
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))

    return cap
  
  



def frame_generator(cap: cv2.VideoCapture, resize: Optional[Tuple[int, int]] = None, flip_horizontal: bool = False) -> Iterator[Tuple[int, np.ndarray]]:
    """Yield frames from an already-opened VideoCapture.

    Single responsibility: read frames, apply optional resize/flip, and yield
    (frame_id, frame). This function does NOT display, log, or release the
    capture device.

    Parameters
    ----------
    cap : cv2.VideoCapture
        An opened VideoCapture object (returned by `openStream`). Caller must
        ensure `cap.isOpened()` before calling.
    resize : Optional[Tuple[int,int]]
        If provided, frames will be resized to (width, height) before yielding.
    flip_horizontal : bool
        If True, frames are horizontally flipped (useful for webcam mirroring).

    Yields
    ------
    Iterator[Tuple[int, np.ndarray]]
        Tuples of (frame_id, frame) where frame_id starts at 0 and increments
        by 1 for each yielded frame. Frame is a BGR `np.ndarray` (OpenCV format).

    Notes
    -----
    - The generator will stop when `cap.read()` returns False.
    - The function does not call `cap.release()`; resource management is the
      responsibility of the caller.
    """
    frame_id = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if flip_horizontal:
            frame = cv2.flip(frame, 1)

        if resize is not None:
            width, height = resize
            frame = cv2.resize(frame, (int(width), int(height)), interpolation=cv2.INTER_LINEAR)

        yield frame_id, frame
        frame_id += 1


# Usage example (for notebook cell):
#
# from openStream import openStream
# from frame_generator import frame_generator
# cap = openStream()\#
# try:
#     for fid, frame in frame_generator(cap, resize=(640,480), flip_horizontal=True):
#         # process frame (e.g., pass to model)
#         if fid > 300:
#             break
# finally:
#     cap.release()
#     cv2.destroyAllWindows()


